### Introduction

Dans ce notebook, nous allons présenter le processus de sélection des variables pour le modèle de prédiction des prix des maisons. En combinant des approches statistiques et des techniques issues de différents algorithmes de machine learning (ElasticNet, RandomForest, XGBoost), nous avons identifié les features les plus importantes. Cette sélection s'est également appuyée sur une compréhension métier et sur l'objectif de maintenir un modèle simple et stable.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OrdinalEncoder

In [2]:
import sys
import os

# Ajouter le chemin du projet à sys.path pour que Python puisse trouver utils
sys.path.append('chemin_vers_le_projet/Housing_price_challenge')
import utils.data_utils as du
import utils.feature_selection as fs

In [3]:
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

In [4]:
from sklearn.linear_model import ElasticNet

In [5]:
path = "chemin_vers_le_projet/Housing_price_challenge/home-data-for-ml-course/train.csv"

In [6]:
houses = du.prep_data(source=path, encoding='ordinal', replace='cat')

/Users/alexandreseverien/projets_ML/Housing_Price_ML/Housing_price_challenge/utils/data_utils.py:196: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  dataframe[col].replace(replacement, inplace=True)
/Users/alexandreseverien/projets_ML/Housing_Price_ML/Housing_price_challenge/utils/data_utils.py:198: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  dataframe[col] = pd.to_numeric(dataframe[col], errors='ignore')
/User

In [7]:
houses = du.convert_and_display(houses)


--- Colonnes discrètes converties ---
BedroomAbvGr - Type: int64 - Exemples: [3 4 1 2 0]
BldgType - Type: int64 - Exemples: [0 1 2 4 3]
BsmtFinType1 - Type: int64 - Exemples: [5 4 0 2 3]
BsmtFullBath - Type: int64 - Exemples: [1 0 2 3]
BsmtQual - Type: int64 - Exemples: [ 94  84 100   0  74]
Condition1 - Type: int64 - Exemples: [1 0 2 3]
ExterQual - Type: int64 - Exemples: [3 1 4]
Exterior1st - Type: int64 - Exemples: [3 2 0 1]
Exterior2nd - Type: int64 - Exemples: [3 2 1 4 0]
Fireplaces - Type: int64 - Exemples: [0 1 2 3]
Foundation - Type: int64 - Exemples: [3 1 0 2]
FullBath - Type: int64 - Exemples: [2 1 3 0]
GarageCars - Type: int64 - Exemples: [2 3 1 0 4]
GarageFinish - Type: int64 - Exemples: [1 2 0 3]
GarageType - Type: int64 - Exemples: [0 2 1 3 4]
HalfBath - Type: int64 - Exemples: [1 0 2]
HeatingQC - Type: int64 - Exemples: [5 4 3 1]
HouseStyle - Type: int64 - Exemples: [1 0]
KitchenQual - Type: int64 - Exemples: [4 2 5]
LotShape - Type: int64 - Exemples: [1 0]
MSSubClass -

In [8]:
X = houses.drop('SalePrice', axis=1)

In [9]:
y = houses.SalePrice

In [10]:
discrete_features = X.dtypes == int

In [11]:
# 1. Stocker les résultats de chaque test dans des variables

# ElasticNet
elasticnet_results = fs.coeff_elastic_net(houses)[0].head(10)
elasticnet_top10 = elasticnet_results['Feature'].values

# Random Forest
rf_top10 = fs.feature_importance_scoring(data=houses, model='RF').head(10).index

# XGBoost
xgb_top10 = fs.feature_importance_scoring(data=houses, model='XGB').head(10).index

# Mutual Information
mi_top10 = fs.make_mi_scores(X, y, discrete_features=discrete_features).head(10).index

# Sélectionner les variables continues
continuous_features = houses.select_dtypes(include='float').columns

# Calculer les corrélations avec 'SalePrice' et stocker les 10 premières
correlations = houses[continuous_features].corrwith(houses['SalePrice']).sort_values(ascending=False)
correlations_top10 = correlations.head(10).index

# ANOVA
anova_results = fs.select_k_best_anova(houses, k='all').head(10)
anova_top10 = anova_results['Feature'].values

# Kendall Correlation
kendall_results = fs.select_kendall_correlation(houses).head(10)
kendall_top10 = kendall_results['Feature'].values

# 2. Créer le dataframe avec les résultats
df = pd.DataFrame({
    'ElasticNet': elasticnet_top10,
    'ANOVA': anova_top10,
    'RandomForest': rf_top10,
    'XGBoost': xgb_top10,
    'MutualInformation': mi_top10,
    'Correlation': correlations_top10,
    'KendallCorrelation': kendall_top10
})

display(df)

Fitting 5 folds for each of 9 candidates, totalling 45 fits
Meilleur score R² sur l'ensemble de test : 0.8314
Meilleur score MAE sur l'ensemble de test : 21915.4841
Meilleurs hyperparamètres : {'elastic_net__alpha': 1.0, 'elastic_net__l1_ratio': 0.9}


,ElasticNet,ANOVA,RandomForest,XGBoost,MutualInformation,Correlation,KendallCorrelation
0,OverallQual,OverallQual,OverallQual,OverallQual,OverallQual,GrLivArea,GrLivArea
1,GrLivArea,ExterQual,GrLivArea,GarageCars,GrLivArea,GarageArea,GarageArea
2,Neighborhood,GarageCars,TotalBsmtSF,KitchenQual,Neighborhood,TotalBsmtSF,YearBuilt
3,KitchenQual,KitchenQual,1stFlrSF,GrLivArea,TotalBsmtSF,1stFlrSF,GarageYrBlt
4,ExterQual,FullBath,BsmtFinSF1,TotRmsAbvGrd,GarageArea,YearBuilt,TotalBsmtSF
5,GarageCars,BsmtQual,LotArea,BsmtQual,GarageCars,GarageYrBlt,YearRemodAdd
6,1stFlrSF,GarageFinish,GarageCars,TotalBsmtSF,YearBuilt,YearRemodAdd,1stFlrSF
7,TotRmsAbvGrd,Neighborhood,GarageArea,Neighborhood,BsmtQual,MasVnrArea,OpenPorchSF
8,2ndFlrSF,TotRmsAbvGrd,2ndFlrSF,1stFlrSF,GarageYrBlt,BsmtFinSF1,MasVnrArea
9,Fireplaces,Foundation,YearBuilt,GarageFinish,ExterQual,LotFrontage,LotArea


### Conclusion

À l'issue des différentes analyses, nous avons retenu les variables suivantes comme étant les plus pertinentes pour le modèle final : 

- **OverallQual** : Qualité globale de la maison
- **GrLivArea** : Surface habitable au-dessus du niveau du sol
- **GarageCars** : Capacité du garage en nombre de voitures
- **TotalBsmtSF** : Surface totale du sous-sol
- **YearBuilt** : Année de construction
- **Neighborhood** : Quartier de la maison

Ce choix a été guidé par plusieurs critères :
- L'importance des variables identifiée par des techniques telles que les coefficients de l'ElasticNet ou les features importances de RandomForest et XGBoost.
- La pertinence métier des variables dans le contexte de la prédiction immobilière.
- Une volonté de simplifier le modèle pour favoriser sa stabilité et éviter le sur-apprentissage.

Ce sous-ensemble de features sera utilisé pour entraîner et évaluer les performances du modèle final dans le notebook suivants.